# Lesson 05: 投資判断整理AI

このノートブックでは、Claude API を使って **投資判断整理AI** を構築します。

**目的**
- 相場分析・トレード判断・リスク確認を整理する
- 感情的な売買を防ぐ

**回答フォーマット（8項目）**
1. 相場環境
2. 上昇要因
3. 下落要因
4. 今の局面評価
5. エントリーするなら条件
6. 見送るなら理由
7. 損切り・利確の考え方
8. 総合判断

---

## 事前準備

```bash
pip install anthropic
```

`ANTHROPIC_API_KEY` 環境変数にAPIキーを設定してください。

In [ ]:
# !pip install anthropic  # 必要に応じてアンコメントして実行
import anthropic
import os

# クライアント初期化（ANTHROPIC_API_KEY 環境変数を使用）
client = anthropic.Anthropic()

print("✅ Anthropic クライアントの初期化が完了しました")

## システムプロンプトの定義

投資判断整理AIのキャラクター・方針・回答形式を定義します。

In [ ]:
SYSTEM_PROMPT = """
あなたは私専用の投資判断整理AIです。
目的は、相場分析、トレード判断、リスク確認を整理して、感情的な売買を防ぐことです。

基本方針:
- 結論を急がず、まず環境認識を整理する
- 強気と弱気の両面を必ず出す
- エントリーよりも見送り判断を大切にする
- リスクリワードと損切り根拠が弱い場合は慎重判断とする
- 不確実な情報は不確実と明記する
- ルール違反の可能性があれば指摘する

回答形式:
必ず以下の8項目を見出し付きで回答すること。

1. 相場環境
2. 上昇要因
3. 下落要因
4. 今の局面評価
5. エントリーするなら条件
6. 見送るなら理由
7. 損切り・利確の考え方
8. 総合判断
""".strip()

print("✅ システムプロンプトを設定しました")

## 投資判断整理AI 関数

Claude API を呼び出して、8項目フォーマットで相場分析を返す関数を定義します。

In [ ]:
def analyze_investment(query: str, verbose: bool = True) -> str:
    """
    投資判断を整理して8項目フォーマットで返す。

    Parameters
    ----------
    query : str
        銘柄・状況・質問などを自由記述で入力。
    verbose : bool
        True のとき結果をセル内に表示する。

    Returns
    -------
    str
        Claude の回答テキスト。
    """
    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=4096,
        thinking={"type": "adaptive"},   # 複雑な判断には adaptive thinking が有効
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": query}
        ],
    )

    # テキストブロックだけ抽出
    answer = "".join(
        block.text
        for block in response.content
        if block.type == "text"
    )

    if verbose:
        print(answer)

    return answer


print("✅ analyze_investment() 関数を定義しました")

## 使い方

下のセルの `query` に銘柄や状況を書いて実行するだけです。

**入力例**
- `"AAPL の現状を分析して。直近で決算があり、株価は高値圏にある。"`
- `"USD/JPY が 155 円を突破。日銀介入リスクを踏まえてエントリー判断して。"`
- `"ビットコインが60,000ドルに接近中。短期トレードを検討している。"`

In [ ]:
# ▼ ここに分析したい銘柄・状況を入力 ▼
query = """
AAPL（アップル）について投資判断を整理してほしい。
状況:
- 株価は直近1ヶ月で +8% 上昇し、高値圏（52週高値付近）にある。
- 翌週に四半期決算発表が予定されている。
- 米国10年国債利回りが上昇傾向にある（リスクオフ懸念）。
- 自分のルール: 決算週はポジションを持ち越さない。
""".strip()

result = analyze_investment(query)

---

## マルチターン対話（会話継続）

同じ銘柄について追加質問したい場合は、会話履歴を保持する `InvestmentAdvisor` クラスが便利です。

In [ ]:
class InvestmentAdvisor:
    """
    会話履歴を保持する投資判断整理AIクライアント。
    同じ銘柄や状況について追加質問できる。
    """

    def __init__(self):
        self.messages: list[dict] = []

    def chat(self, user_input: str, verbose: bool = True) -> str:
        self.messages.append({"role": "user", "content": user_input})

        response = client.messages.create(
            model="claude-opus-4-6",
            max_tokens=4096,
            thinking={"type": "adaptive"},
            system=SYSTEM_PROMPT,
            messages=self.messages,
        )

        answer = "".join(
            block.text
            for block in response.content
            if block.type == "text"
        )

        # アシスタントの返答を履歴に追加（テキストのみ保存）
        self.messages.append({"role": "assistant", "content": answer})

        if verbose:
            print(answer)

        return answer

    def reset(self):
        """会話履歴をリセットする。"""
        self.messages = []
        print("🔄 会話履歴をリセットしました")


# インスタンス生成
advisor = InvestmentAdvisor()
print("✅ InvestmentAdvisor を初期化しました")

In [ ]:
# 1回目: 初期分析
_ = advisor.chat("""
USD/JPY について判断を整理してほしい。
現在レート: 152.80円
状況: 米FOMC議事録で利下げ慎重姿勢が示され、ドル高継続中。
日銀は現状維持を示唆。短期でのロングエントリーを検討中。
""".strip())

In [ ]:
# 2回目: 追加質問（会話継続）
_ = advisor.chat("損切りラインを 152.00 に設定した場合、リスクリワードは許容範囲か？")

In [ ]:
# 会話履歴をリセットして新しい銘柄を分析
advisor.reset()

---

## まとめ

| 機能 | 方法 |
|------|------|
| 単発分析 | `analyze_investment(query)` |
| 会話継続 | `advisor.chat(query)` |
| 履歴リセット | `advisor.reset()` |

**ポイント**
- `thinking={"type": "adaptive"}` により、複雑な状況ほどモデルが深く思考してから回答します。
- システムプロンプトで「強気・弱気の両面提示」「見送り重視」を徹底させているため、過度にポジティブな分析を抑制できます。
- あくまで補助ツールです。最終的な投資判断は自己責任で行ってください。